In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


## use biopython read fasta

In [ ]:
from Bio import SeqIO

fasta_file = our_data + "Hypericum_PF00067_Merge97.pep"

In [ ]:
sequences = []
for record in SeqIO.parse(fasta_file, "fasta"):
    sequences.append([record.id, str(record.seq)])

with open(our_data + "Hypericum_PF00067_Merge97.fasta", "w") as output_handle:
    for seq_id, seq_data in sequences:
        output_handle.write(f">{seq_id}\n{seq_data}\n")

df_enzymedata = pd.DataFrame(sequences, columns=["enzyme", "Sequence"])

In [ ]:
df_enzymedata.to_csv(
    our_data + "Digitalis_leaf_PF00067_Merge98_Sort_Express.csv", index=False
)
df_enzymedata

,enzyme,Sequence
0,Hste_113815,MNLLILLLLSLILVLVFYFTYSLIWVPLRVQAHFRAQGVTGPGFRP...
1,Hlea_018408,MPDTIQCLNPFPSFATTNIYKPTDKKTMDQFYLTLLLLFVSSIALS...
2,Hflo_189074,MDGLYLKLALTLLLSIFIAYKLFFRKKHNYPPGPRALPIIGNLHLI...
3,Hflo_160077,MIMSPFVLYSFVVAVLVYFLVSLKSALSGRRLPPGPKPWPIVGNLP...
4,Hlea_230320,MVHSITKIEQEMLSASVFLAGATAIALILLFFWNSKRSSDKTSRLP...
...,...,...
321,Hflo_298643,MGFFHLLSLAIVGVFLYGFFKFLISCWVSPTRAYLKLRRNGFGGPT...
322,Hflo_264086,MDVSTALMLVAGVTAYLLWFTFISRTLRGPRVWPLLGSLPGLIENC...
323,Hflo_057832,MDVSTALMLVAGVTAYLLWFTFISRTLRGPRVWPLLGSLPGLIENC...
324,Hflo_121886,MELVTIFTLIVCVLTLLTIIRTISRSSYKNGGAKLPPGPSPLPVLG...


In [ ]:
data = {
    "substrate": ["1,3,5-trihydroxyxanthone", "1,3,7-trihydroxyxanthone"],
    "inchi": [
        "InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-7(12(11)17)2-1-3-8(13)15/h1-5,14-16H",
        "InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14(18)13-9(17)2-6(15)3-12(13)20-10/h2-5,15-17H,1H3",
    ],
}

df_substratedata = pd.DataFrame(data)
df_substratedata

,substrate,inchi
0,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...
1,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...


In [ ]:
df_enzymedata["key"] = 1
df_substratedata["key"] = 1
result = pd.merge(df_enzymedata, df_substratedata, on="key", how="outer")

result.drop(columns="key", inplace=True)

In [7]:
result

,enzyme,Sequence,substrate,inchi
0,Hste_113815,MNLLILLLLSLILVLVFYFTYSLIWVPLRVQAHFRAQGVTGPGFRP...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...
1,Hste_113815,MNLLILLLLSLILVLVFYFTYSLIWVPLRVQAHFRAQGVTGPGFRP...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...
2,Hlea_018408,MPDTIQCLNPFPSFATTNIYKPTDKKTMDQFYLTLLLLFVSSIALS...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...
3,Hlea_018408,MPDTIQCLNPFPSFATTNIYKPTDKKTMDQFYLTLLLLFVSSIALS...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...
4,Hflo_189074,MDGLYLKLALTLLLSIFIAYKLFFRKKHNYPPGPRALPIIGNLHLI...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...
...,...,...,...,...
647,Hflo_057832,MDVSTALMLVAGVTAYLLWFTFISRTLRGPRVWPLLGSLPGLIENC...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...
648,Hflo_121886,MELVTIFTLIVCVLTLLTIIRTISRSSYKNGGAKLPPGPSPLPVLG...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...
649,Hflo_121886,MELVTIFTLIVCVLTLLTIIRTISRSSYKNGGAKLPPGPSPLPVLG...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...
650,Hflo_274570,MINNIMLSLLSLACYIRQTNFITLKLLFNHLRPLMLVSLALTTAFL...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...


In [10]:
# ! python extract.py esm1b_t33_650M_UR50S {our_data}'Hypericum_PF00067_Merge97.fasta' {our_data}esm --repr_layers 33 --include mean

Transferred model to GPU
Read /home/hanxd/Repositories/ESP/our_codes/../data/our_data/Hypericum_PF00067_Merge97.fasta with 326 sequences
Processing 1 of 44 batches (8 sequences)
Processing 2 of 44 batches (8 sequences)
Processing 3 of 44 batches (8 sequences)
Processing 4 of 44 batches (8 sequences)
Processing 5 of 44 batches (8 sequences)
Processing 6 of 44 batches (8 sequences)
Processing 7 of 44 batches (8 sequences)
Processing 8 of 44 batches (8 sequences)
Processing 9 of 44 batches (8 sequences)
Processing 10 of 44 batches (8 sequences)
Processing 11 of 44 batches (8 sequences)
Processing 12 of 44 batches (8 sequences)
Processing 13 of 44 batches (8 sequences)
Processing 14 of 44 batches (8 sequences)
Processing 15 of 44 batches (8 sequences)
Processing 16 of 44 batches (8 sequences)
Processing 17 of 44 batches (8 sequences)
Processing 18 of 44 batches (8 sequences)
Processing 19 of 44 batches (8 sequences)
Processing 20 of 44 batches (8 sequences)
Processing 21 of 44 batches (8 s

In [ ]:
result["ESM1b"] = ""
result["ECFP"] = ""

rep_dict = join(CURRENT_DIR, "..", "data", "our_data", "esm/")

for ind in result.index:
    esms = torch.load(rep_dict + str(result["enzyme"][ind]) + ".pt")
    ecfps = Chem.MolFromInchi(result["inchi"][ind])
    ecfpso = AllChem.GetMorganFingerprintAsBitVect(
        ecfps, radius=2, nBits=1024
    ).ToBitString()
    result["ESM1b"][ind] = esms["mean_representations"][33].tolist()
    result["ECFP"][ind] = ecfpso
result["Binding"] = 0

In [12]:
result

,enzyme,Sequence,substrate,inchi,ESM1b,ECFP,Binding
0,Hste_113815,MNLLILLLLSLILVLVFYFTYSLIWVPLRVQAHFRAQGVTGPGFRP...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...,"[-0.07908860594034195, 0.20324212312698364, 0....",0000000000000001000000000000000000000000000000...,0
1,Hste_113815,MNLLILLLLSLILVLVFYFTYSLIWVPLRVQAHFRAQGVTGPGFRP...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...,"[-0.07908860594034195, 0.20324212312698364, 0....",0000000000000000000000000000000001000000100000...,0
2,Hlea_018408,MPDTIQCLNPFPSFATTNIYKPTDKKTMDQFYLTLLLLFVSSIALS...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...,"[-0.1608358770608902, 0.24539121985435486, 0.0...",0000000000000001000000000000000000000000000000...,0
3,Hlea_018408,MPDTIQCLNPFPSFATTNIYKPTDKKTMDQFYLTLLLLFVSSIALS...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...,"[-0.1608358770608902, 0.24539121985435486, 0.0...",0000000000000000000000000000000001000000100000...,0
4,Hflo_189074,MDGLYLKLALTLLLSIFIAYKLFFRKKHNYPPGPRALPIIGNLHLI...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...,"[-0.01506323553621769, 0.12608227133750916, -0...",0000000000000001000000000000000000000000000000...,0
...,...,...,...,...,...,...,...
647,Hflo_057832,MDVSTALMLVAGVTAYLLWFTFISRTLRGPRVWPLLGSLPGLIENC...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...,"[-0.1442718654870987, 0.05491785705089569, 0.1...",0000000000000000000000000000000001000000100000...,0
648,Hflo_121886,MELVTIFTLIVCVLTLLTIIRTISRSSYKNGGAKLPPGPSPLPVLG...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...,"[-0.022586463019251823, 0.11354727298021317, -...",0000000000000001000000000000000000000000000000...,0
649,Hflo_121886,MELVTIFTLIVCVLTLLTIIRTISRSSYKNGGAKLPPGPSPLPVLG...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...,"[-0.022586463019251823, 0.11354727298021317, -...",0000000000000000000000000000000001000000100000...,0
650,Hflo_274570,MINNIMLSLLSLACYIRQTNFITLKLLFNHLRPLMLVSLALTTAFL...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...,"[-0.009770087897777557, 0.18663787841796875, 0...",0000000000000001000000000000000000000000000000...,0


In [ ]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)

        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y)


feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

data_X, data_y = create_input_and_output_data(df=result)

In [ ]:
bst = pd.read_pickle(
    os.path.join(CURRENT_DIR, "..", "data", "our_data", "p450normalmodel.dat")
)
dwant = xgb.DMatrix(
    np.array(data_X), label=np.array(data_y), feature_names=feature_names
)
y_test_pred = bst.predict(dwant)
result["scores"] = y_test_pred

In [15]:
result

,enzyme,Sequence,substrate,inchi,ESM1b,ECFP,Binding,scores
0,Hste_113815,MNLLILLLLSLILVLVFYFTYSLIWVPLRVQAHFRAQGVTGPGFRP...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...,"[-0.07908860594034195, 0.20324212312698364, 0....",0000000000000001000000000000000000000000000000...,0,0.010890
1,Hste_113815,MNLLILLLLSLILVLVFYFTYSLIWVPLRVQAHFRAQGVTGPGFRP...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...,"[-0.07908860594034195, 0.20324212312698364, 0....",0000000000000000000000000000000001000000100000...,0,0.033392
2,Hlea_018408,MPDTIQCLNPFPSFATTNIYKPTDKKTMDQFYLTLLLLFVSSIALS...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...,"[-0.1608358770608902, 0.24539121985435486, 0.0...",0000000000000001000000000000000000000000000000...,0,0.003406
3,Hlea_018408,MPDTIQCLNPFPSFATTNIYKPTDKKTMDQFYLTLLLLFVSSIALS...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...,"[-0.1608358770608902, 0.24539121985435486, 0.0...",0000000000000000000000000000000001000000100000...,0,0.017599
4,Hflo_189074,MDGLYLKLALTLLLSIFIAYKLFFRKKHNYPPGPRALPIIGNLHLI...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...,"[-0.01506323553621769, 0.12608227133750916, -0...",0000000000000001000000000000000000000000000000...,0,0.931864
...,...,...,...,...,...,...,...,...
647,Hflo_057832,MDVSTALMLVAGVTAYLLWFTFISRTLRGPRVWPLLGSLPGLIENC...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...,"[-0.1442718654870987, 0.05491785705089569, 0.1...",0000000000000000000000000000000001000000100000...,0,0.015511
648,Hflo_121886,MELVTIFTLIVCVLTLLTIIRTISRSSYKNGGAKLPPGPSPLPVLG...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...,"[-0.022586463019251823, 0.11354727298021317, -...",0000000000000001000000000000000000000000000000...,0,0.778912
649,Hflo_121886,MELVTIFTLIVCVLTLLTIIRTISRSSYKNGGAKLPPGPSPLPVLG...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...,"[-0.022586463019251823, 0.11354727298021317, -...",0000000000000000000000000000000001000000100000...,0,0.404401
650,Hflo_274570,MINNIMLSLLSLACYIRQTNFITLKLLFNHLRPLMLVSLALTTAFL...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...,"[-0.009770087897777557, 0.18663787841796875, 0...",0000000000000001000000000000000000000000000000...,0,0.403449


In [ ]:
result.to_csv(our_data + "onetest_cheng2.csv")

In [ ]:
pivot_df = result.pivot(index="enzyme", columns="substrate", values="scores")

In [ ]:
pivot_df.to_csv(our_data + "onetest_cheng2_restruate.csv")

In [ ]:
result[result["enzyme"] == "Hflo_274570"]

,enzyme,Sequence,substrate,inchi,ESM1b,ECFP,Binding,scores
650,Hflo_274570,MINNIMLSLLSLACYIRQTNFITLKLLFNHLRPLMLVSLALTTAFL...,"1,3,5-trihydroxyxanthone",InChI=1S/C13H8O5/c14-6-4-9(16)11-10(5-6)18-13-...,"[-0.009770087897777557, 0.18663787841796875, 0...",0000000000000001000000000000000000000000000000...,0,0.403449
651,Hflo_274570,MINNIMLSLLSLACYIRQTNFITLKLLFNHLRPLMLVSLALTTAFL...,"1,3,7-trihydroxyxanthone",InChI=1S/C14H10O6/c1-19-11-5-10-7(4-8(11)16)14...,"[-0.009770087897777557, 0.18663787841796875, 0...",0000000000000000000000000000000001000000100000...,0,0.540107
